# Equity Duration — Clean Notebook

This notebook constructs several **duration** measures for STOXX Europe 600 constituents:

1. **Operating cash-flow timing duration** (undiscounted)
2. **Discounted cash-flow duration** with fixed discount rate \(r = 12.5\%\)
3. **Discounted cash-flow duration** with firm-specific **CAPM cost of equity**
4. **Empirical (interest-rate sensitivity) duration** using daily changes in **2Y EUR OIS** (RIC `EUREON2Y=`)

> **Note on cash flows:** `CFPSMean_*` in this project is *cash flow from operations per share* (CFO per share), not net payouts to equity holders.


## 0. Setup

- Requires: `lseg.data` (LSEG Data Library), `pandas`, `numpy`
- You may need to authenticate / configure LSEG before running.


In [ ]:
import numpy as np
import pandas as pd
import lseg.data as ld
from pathlib import Path

#Speicherstruktur für Intermediate und Final Output
BASE_DIR = Path(
    "/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data"
)
BASE_DIR.mkdir(parents=True, exist_ok=True)

(TABLE_DIR := BASE_DIR / "tables").mkdir(exist_ok=True)
(DATA_DIR  := BASE_DIR / "intermediate").mkdir(exist_ok=True)

def save_parquet(df: pd.DataFrame, name: str):
    path = DATA_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved: {path}")

def load_parquet(name: str) -> pd.DataFrame:
    path = DATA_DIR / f"{name}.parquet"
    return pd.read_parquet(path)


## 1. Load the STOXX 600 constituent list

Expected columns (names can vary):
- RIC
- Company name
- Country of headquarters
- Market cap

Adjust `INPUT_XLSX` if needed.


In [23]:
# --- User input ---
INPUT_XLSX = DATA_DIR / "STOXX600Constituents202601.xlsx"
SHEET_NAME = 0  # or a sheet name string

df_const = pd.read_excel(INPUT_XLSX, sheet_name=SHEET_NAME)

# Robust column detection
ric_col = next(c for c in df_const.columns if "RIC" in c.upper())
name_col = next(c for c in df_const.columns if "COMPANY" in c.upper() or "NAME" in c.upper())
country_col = next(c for c in df_const.columns if "COUNTRY" in c.upper())
mcap_col = next(c for c in df_const.columns if "MARKET" in c.upper() and "CAP" in c.upper())

df_basic = df_const[[ric_col, name_col, country_col, mcap_col]].copy()
df_basic.columns = ["RIC", "CompanyName", "CountryHQ", "MarketCap"]

df_basic = (
    df_basic
    .dropna(subset=["RIC"])
    .drop_duplicates(subset=["RIC"])
    .sort_values("RIC")
    .reset_index(drop=True)
)

print(df_basic.head())
print("Number of firms:", len(df_basic))


       RIC         CompanyName               CountryHQ     MarketCap
0    A2.MI             A2A SpA                 Italien   7610.187355
1    AAF.L   Airtel Africa PLC  Vereinigtes Königreich  15313.262664
2   AAK.ST       AAK AB (publ)                Schweden   6142.691574
3    AAL.L  Anglo American PLC  Vereinigtes Königreich  42523.693525
4  AALB.AS         Aalberts NV             Niederlande   3222.012813
Number of firms: 600


## 2. Helper functions (LSEG pulls + finance math)

We define small helper functions for:
- Year-end price
- ROE (FY0)
- 5-year beta
- CAPM cost of equity
- Duration measures (discounted + undiscounted)


In [20]:
def get_year_end_price(ric: str, year: int = 2025, field: str = "TR.PriceClose") -> float:
    """Get the last available daily price in a given calendar year."""
    hist = ld.get_history(
        universe=[ric],
        fields=[field],
        start=f"{year}-01-01",
        end=f"{year}-12-31",
        interval="daily"
    )
    if hist is None or hist.empty:
        return np.nan

    # hist typically has Date as index and one column for the field
    df = hist.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index()
        date_col = next(c for c in df.columns if "date" in str(c).lower())
        df[date_col] = pd.to_datetime(df[date_col])
        df = df.set_index(date_col)

    # take last non-missing value
    s = pd.to_numeric(df.iloc[:, 0], errors="coerce").dropna()
    return float(s.iloc[-1]) if len(s) else np.nan


def get_roe_fy0(ric: str) -> float:
    """ROE (FY0) from LSEG. Returns decimal (0.15 = 15%)."""
    roe_df = ld.get_data(
        universe=[ric],
        fields=["TR.F.ReturnAvgTotEqPct(period=FY0)"]
    )
    if roe_df is None or roe_df.empty:
        return np.nan

    roe_col = next((c for c in roe_df.columns if "Return on Average Total Equity" in c), None)
    if roe_col is None:
        # fallback: take the only non-id column
        roe_col = roe_df.columns[-1]

    roe = roe_df[roe_col].iloc[0]
    if roe is None:
        return np.nan

    return float(roe) / 100.0


def get_beta_5y(ric: str) -> float:
    """5Y beta from LSEG."""
    beta_df = ld.get_data(universe=[ric], fields=["TR.BetaFiveYear"])
    if beta_df is None or beta_df.empty:
        return np.nan

    beta_col = next((c for c in beta_df.columns if "Beta" in c), None)
    if beta_col is None:
        beta_col = beta_df.columns[-1]

    b = beta_df[beta_col].iloc[0]
    return float(b) if b is not None else np.nan


def coe_capm(beta: float, rf: float, erp: float) -> float:
    """CAPM cost of equity in decimals."""
    if beta is None or (isinstance(beta, float) and np.isnan(beta)):
        return np.nan
    return rf + beta * erp


def duration_discounted(CF: np.ndarray, P: float, r: float) -> float:
    """Implied duration with terminal value inferred from price."""
    if not np.isfinite(P) or P <= 0 or not np.isfinite(r) or r <= 0:
        return np.nan
    CF = np.asarray(CF, dtype=float)
    CF = CF[np.isfinite(CF)]
    T = len(CF)
    if T < 2:
        return np.nan

    t = np.arange(1, T + 1, dtype=float)
    pv_cf = CF / (1.0 + r) ** t
    pv_sum = pv_cf.sum()

    term1 = (t * pv_cf).sum() / P
    term2 = (T + (1.0 + r) / r) * ((P - pv_sum) / P)
    return float(term1 + term2)


def duration_undiscounted(CF: np.ndarray) -> float:
    """Cash-flow timing duration (undiscounted)."""
    CF = np.asarray(CF, dtype=float)
    CF = CF[np.isfinite(CF)]
    T = len(CF)
    if T < 2:
        return np.nan
    t = np.arange(1, T + 1, dtype=float)
    return float((t * CF).sum() / CF.sum()) if CF.sum() != 0 else np.nan


## 3. Pull firm-level inputs (price, ROE, beta) and compute CAPM \(r_i\)

- **Price (Dec 31, 2025):** last available daily close in 2025
- **ROE FY0:** used for descriptive stats (not for duration construction here)
- **Beta (5Y):** used for CAPM \(r_i\)


In [ ]:
# --- User choices for CAPM ---
RF = 0.025   # 2.5% risk-free (decimal)
ERP = 0.05   # 5% equity risk premium (decimal)

ld.open_session()

# Year-end price
prices = []
for ric in df_basic["RIC"]:
    try:
        prices.append(get_year_end_price(ric, year=2025, field="TR.PriceClose"))
    except Exception:
        prices.append(np.nan)
df_basic["Price_2025_12_31"] = prices

# ROE FY0
roe_vals = []
for ric in df_basic["RIC"]:
    try:
        roe_vals.append(get_roe_fy0(ric))
    except Exception:
        roe_vals.append(np.nan)
df_basic["ROE_FY0"] = roe_vals

# Beta + CAPM CoE
betas = []
coes = []
for ric in df_basic["RIC"]:
    try:
        b = get_beta_5y(ric)
    except Exception:
        b = np.nan
    betas.append(b)
    coes.append(coe_capm(b, rf=RF, erp=ERP))

df_basic["BETA_5Y"] = betas
df_basic["COE_CAPM"] = coes

ld.close_session()

df_basic[["RIC","Price_2025_12_31","ROE_FY0","BETA_5Y","COE_CAPM"]].head()

firm_cols = [
    "RIC", "CompanyName", "CountryHQ",
    "MarketCap", "Price_2025_12_31",
    "ROE_FY0", "BETA_5Y", "COE_CAPM"
]

df_firm_inputs = df_basic[firm_cols].copy()
save_parquet(df_firm_inputs, "firm_inputs")


KeyboardInterrupt: 

## 4. Analyst operating cash-flow forecasts (CFPS)

This notebook expects these columns already in `df_basic`:

- `CFPSMean_FY25` … `CFPSMean_FY29`

If you pull them from LSEG in your project, keep that code in one place and ensure these columns exist before continuing.


In [ ]:
cf_cols = ["CFPSMean_FY25", "CFPSMean_FY26", "CFPSMean_FY27", "CFPSMean_FY28", "CFPSMean_FY29"]

# Ensure numeric
for c in cf_cols:
    df_basic[c] = pd.to_numeric(df_basic[c], errors="coerce")

df_basic[["RIC"] + cf_cols].head()

cf_cols = ["CFPSMean_FY25", "CFPSMean_FY26", "CFPSMean_FY27", "CFPSMean_FY28", "CFPSMean_FY29"]

df_cfps = df_basic[["RIC"] + cf_cols].copy()
save_parquet(df_cfps, "cfps_forecasts")


## 5. Duration measures from CFPS

We compute three timing measures:

- `Duration_r125`: discounted duration with fixed \(r=12.5\%\)
- `Duration_CAPM`: discounted duration with firm-specific \(r_i\) from CAPM
- `Duration_undiscounted`: pure timing (no discounting)

All durations use **price per share** and **CFPS** (per share), so units are consistent.


In [ ]:
R_FIXED = 0.125

# Build CF arrays per row
def row_cf_array(row) -> np.ndarray:
    return np.array([row.get(c) for c in cf_cols], dtype=float)

df_basic["Duration_r125"] = df_basic.apply(
    lambda row: duration_discounted(row_cf_array(row), P=row["Price_2025_12_31"], r=R_FIXED),
    axis=1
)

df_basic["Duration_CAPM"] = df_basic.apply(
    lambda row: duration_discounted(row_cf_array(row), P=row["Price_2025_12_31"], r=row["COE_CAPM"]),
    axis=1
)

df_basic["Duration_undiscounted"] = df_basic.apply(
    lambda row: duration_undiscounted(row_cf_array(row)),
    axis=1
)

df_basic[["RIC","Duration_r125","Duration_CAPM","Duration_undiscounted"]].head()


## 6. Empirical duration (2.2): interest-rate sensitivities

We estimate firm-specific sensitivities to **daily changes in the 2Y EUR OIS** rate:

$
r_{i,t} = a_i + b_i \Delta y_t + \varepsilon_{i,t}
\quad\Rightarrow\quad
D^{emp}_i = -b_i
$

### 6.1 Pull the daily 2Y OIS series (RIC `EUREON2Y=`)

We use the field that is available in your setup: `TR.MIDPRICE` (rate in **percent**).


In [ ]:
ld.open_session()

# 2Y EUR OIS / EONIA-style series
ric_rate = "EUREON2Y="
field_rate = "TR.MIDPRICE"

y_df = ld.get_history(
    universe=[ric_rate],
    fields=[field_rate],
    start="2015-01-01",
    end="2025-12-31",
    interval="daily"
)

ld.close_session()

# Build daily rate changes (in percentage points)
rate_df = y_df.copy().reset_index()
rate_df.columns = ["date", "y"]   # y in percent
rate_df["date"] = pd.to_datetime(rate_df["date"])
rate_df["y"] = pd.to_numeric(rate_df["y"], errors="coerce")
rate_df = rate_df.dropna(subset=["y"]).sort_values("date")
rate_df["dy"] = rate_df["y"].diff()

rates_daily = rate_df[["date","dy"]].dropna()

rates_daily.head()


### 6.2 Pull daily stock prices and compute daily returns

We retrieve daily prices for all RICs, compute simple returns, then merge with `rates_daily`.


In [ ]:
def chunk_list(x, n=40):
    for i in range(0, len(x), n):
        yield x[i:i+n]


def get_prices_daily(rics, start="2015-01-01", end="2025-12-31",
                     field="TR.PriceClose", batch_size=40):
    """Fast pull of daily prices (one field) in batches; returns long format."""
    out = []
    for batch in chunk_list(list(rics), n=batch_size):
        df = ld.get_history(
            universe=batch,
            fields=[field],
            start=start,
            end=end,
            interval="daily"
        )
        if df is None or df.empty:
            continue

        df_w = df.copy()
        if not isinstance(df_w.index, pd.DatetimeIndex):
            df_w = df_w.reset_index()
            date_col = next((c for c in df_w.columns if "date" in str(c).lower()), None)
            df_w[date_col] = pd.to_datetime(df_w[date_col])
            df_w = df_w.set_index(date_col)

        df_long = df_w.stack(dropna=False).reset_index()
        df_long.columns = ["date", "RIC", "price"]
        df_long["date"] = pd.to_datetime(df_long["date"])
        df_long["price"] = pd.to_numeric(df_long["price"], errors="coerce")
        df_long = df_long.dropna(subset=["price"])
        out.append(df_long)

    return pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=["date","RIC","price"])


ld.open_session()
prices_daily = get_prices_daily(df_basic["RIC"].tolist(), start="2015-01-01", end="2025-12-31",
                                field="TR.PriceClose", batch_size=40)
ld.close_session()

# daily returns
prices_daily = prices_daily.sort_values(["RIC","date"])
prices_daily["ret"] = prices_daily.groupby("RIC")["price"].pct_change()
rets_daily = prices_daily.dropna(subset=["ret"])[["date","RIC","ret"]].copy()

rets_daily.head()


### 6.3 Estimate firm-level rate betas and empirical duration

In [ ]:
def estimate_rate_beta_duration(rets_daily: pd.DataFrame, rates_daily: pd.DataFrame, min_obs: int = 200) -> pd.DataFrame:
    """Estimate r_{i,t} = a_i + b_i dy_t; return b_i and D_emp=-b_i."""
    df = rets_daily.merge(rates_daily, on="date", how="inner").dropna(subset=["ret","dy"]).copy()

    out = []
    for ric, g in df.groupby("RIC"):
        if len(g) < min_obs:
            continue
        x = g["dy"].to_numpy(float)   # dy in percentage points
        y = g["ret"].to_numpy(float)

        X = np.column_stack([np.ones_like(x), x])
        alpha, b = np.linalg.lstsq(X, y, rcond=None)[0]
        out.append({"RIC": ric, "beta_rate_2yOIS": b, "D_emp_2yOIS": -b, "n_obs": len(g)})

    return pd.DataFrame(out)

dur_emp = estimate_rate_beta_duration(rets_daily, rates_daily, min_obs=200)
df_basic = df_basic.merge(dur_emp, on="RIC", how="left")

print(df_basic[["RIC","D_emp_2yOIS","beta_rate_2yOIS","n_obs"]].head())
print(df_basic["D_emp_2yOIS"].describe())
print(df_basic[["Duration_r125","D_emp_2yOIS"]].corr(method="spearman"))
print("Share with D_emp:", df_basic["D_emp_2yOIS"].notna().mean())


## 7. Final results table (rounded)

We include:
- Firm descriptors
- CAPM inputs
- CFPS forecasts
- All duration measures (rounded to 2 decimals)


In [ ]:
cols = [
    "RIC","CompanyName","CountryHQ","MarketCap","Price_2025_12_31",
    "ROE_FY0","BETA_5Y","COE_CAPM",
    "CFPSMean_FY25","CFPSMean_FY26","CFPSMean_FY27","CFPSMean_FY28","CFPSMean_FY29",
    "Duration_r125","Duration_CAPM","Duration_undiscounted","D_emp_2yOIS"
]

df_results = df_basic[cols].copy()

# percentages for presentation
df_results["ROE_FY0"] = df_results["ROE_FY0"] * 100
df_results["COE_CAPM"] = df_results["COE_CAPM"] * 100

df_results = df_results.rename(columns={
    "Price_2025_12_31": "Price (Dec 31, 2025)",
    "ROE_FY0": "ROE (%)",
    "BETA_5Y": "Beta (5Y)",
    "COE_CAPM": "Cost of Equity CAPM (%)",
    "CFPSMean_FY25": "CFPS FY25",
    "CFPSMean_FY26": "CFPS FY26",
    "CFPSMean_FY27": "CFPS FY27",
    "CFPSMean_FY28": "CFPS FY28",
    "CFPSMean_FY29": "CFPS FY29",
    "Duration_r125": "Duration (r = 12.5%)",
    "Duration_CAPM": "Duration (CAPM r)",
    "Duration_undiscounted": "Duration (undiscounted)",
    "D_emp_2yOIS": "Empirical Duration (2Y OIS)"
})

df_results_rounded = df_results.copy()

# round most numeric columns
df_results_rounded = df_results_rounded.round({
    "MarketCap": 2,
    "Price (Dec 31, 2025)": 2,
    "ROE (%)": 2,
    "Beta (5Y)": 2,
    "Cost of Equity CAPM (%)": 2,
    "CFPS FY25": 2, "CFPS FY26": 2, "CFPS FY27": 2, "CFPS FY28": 2, "CFPS FY29": 2,
})

# round all duration columns automatically
duration_cols = [c for c in df_results_rounded.columns if "Duration" in c]
df_results_rounded[duration_cols] = df_results_rounded[duration_cols].round(2)

df_results_rounded.sort_values("MarketCap", ascending=False).head(20)

output_path = "final_results_duration.xlsx"
df_results_rounded.to_excel(output_path, index=False)

print(f"Saved to {output_path}")

df_results_rounded.to_excel(TABLE_DIR / "duration_full.xlsx", index=False)


KeyError: "['ROE_FY0', 'BETA_5Y', 'COE_CAPM', 'CFPSMean_FY25', 'CFPSMean_FY26', 'CFPSMean_FY27', 'CFPSMean_FY28', 'CFPSMean_FY29', 'Duration_r125', 'Duration_CAPM', 'Duration_undiscounted', 'D_emp_2yOIS'] not in index"